# llmpic — 自然语言图表生成 SDK

> 说人话，出图表。11 种类型，PNG/SVG/PDF 导出，Jupyter 内联显示。

## 1. 初始化 SDK

In [ ]:
import os
from llmpic import llmPIC

# 从环境变量读取密钥，或直接填
lp = llmPIC(
    api_key=os.environ.get("LLMPIC_API_KEY", "sk-your-key"),
    base_url=os.environ.get("LLMPIC_BASE_URL", "https://api.openai.com/v1"),
    model=os.environ.get("LLMPIC_MODEL", "gpt-4o"),
    max_tokens=4096,
)
print(f"SDK ready — model: {lp.model}")

## 2. 基础图表 — 一句话出图

In [ ]:
r = lp.plot("过去12个月的月度销售额变化趋势").render()

if r.success:
    r.show()  # ← Jupyter 内直接出图
    print(f"tokens: in={r.token_usage['input']}, out={r.token_usage['output']}")
else:
    print(f"失败: {r.error_message}")

## 3. 带数据和样式

In [ ]:
import pandas as pd
import numpy as np

np.random.seed(42)
df = pd.DataFrame({
    "月份": [f"{i}月" for i in range(1, 13)],
    "销售额": np.random.randint(80, 200, 12),
    "利润": np.random.randint(15, 50, 12),
})

r = lp.plot("销售额和利润月度趋势").data(df).style({
    "figsize": [14, 6],
    "color_scheme": "cool",
}).render()

r.show()

## 4. 全部图表类型

In [ ]:
from IPython.display import display, Markdown

charts = [
    ("line",     lp.plot("2024年每月营收趋势")),
    ("scatter",  lp.scatter("随机散点图，50个点")),
    ("bar",      lp.bar("各部门预算: 研发=200, 市场=150, 销售=180, 人事=100")),
    ("pie",      lp.pie("市场份额: A=40%, B=25%, C=20%, 其他=15%")),
    ("hist",     lp.hist("正态分布数据，均值0标准差1，1000个样本")),
    ("heatmap",  lp.heatmap("5x5 相关性矩阵")),
    ("boxplot",  lp.boxplot("A/B/C/D 四组实验数据分布")),
    ("area",     lp.area("2020-2024各产品线收入趋势")),
    ("radar",    lp.radar("产品评分: 性能4,易用3,稳定5,价格2,售后4")),
    ("subplots", lp.subplots("2x2看板: 折线,柱状,散点,饼图")),
]

for name, builder in charts:
    r = builder.render()
    if r.success:
        display(Markdown(f"### {name}"))
        r.show()
    else:
        display(Markdown(f"### {name} ✗ — {r.error_message[:100]}"))

## 5. 迭代编辑 — 不满意就接着说

In [ ]:
from IPython.display import display, Markdown

v1 = lp.plot("季度销售: Q1=100, Q2=150, Q3=120, Q4=180").render()
display(Markdown("### v1 — 初始"))
v1.show()

v2 = v1.edit("改成柱状图，柱子颜色换成蓝色系")
display(Markdown("### v2 — 改柱状图+蓝色"))
v2.show()

v3 = v2.edit("标题改为'2025年度销售报告'，字号增大，加网格")
display(Markdown("### v3 — 改标题+样式"))
v3.show()

## 6. 多格式导出 & 保存

In [ ]:
r = lp.plot("sin(x) from 0 to 2π").render()

# 一个 save() 搞定，格式由扩展名自动决定
r.save("demo_chart.png")   # PNG
r.save("demo_chart.svg")   # SVG 矢量
r.save("demo_chart.pdf")   # PDF 打印

# 不传路径 → 默认保存到 ~/llmpic_charts/chart_{timestamp}.png
r.save()

# base64 嵌入 HTML
print(f"PNG base64 长度: {len(r.base64())} chars")
print(f"SVG base64 长度: {len(r.base64_svg())} chars")

## 7. 异步批量生成

In [ ]:
from llmpic import AsyncllmPIC
from IPython.display import display, Markdown
import time

async def run_batch():
    lp_async = AsyncllmPIC(
        api_key=os.environ.get("LLMPIC_API_KEY", "sk-your-key"),
        base_url=os.environ.get("LLMPIC_BASE_URL", "https://api.openai.com/v1"),
        model=os.environ.get("LLMPIC_MODEL", "gpt-4o"),
    )
    
    t0 = time.time()
    results = await lp_async.batch([
        ("plot", "全国12个月销售趋势"),
        ("bar", "各地区销售额对比"),
        ("pie", "市场份额分布"),
        ("scatter", "客户年龄vs消费金额"),
    ])
    
    for i, r in enumerate(results):
        if r.success:
            display(Markdown(f"### batch[{i}] ✓"))
            r.show()
        else:
            print(f"[{i}] 失败: {r.error_message[:100]}")
    
    print(f"4张图并发生成，总耗时 {time.time()-t0:.1f}s")

await run_batch()